> 랭체인 공식 문서 Tools: <https://docs.langchain.com/oss/python/langchain/tools>

### 도구 호출 에이전트(Tool Calling Agent)

In [10]:
from dotenv import load_dotenv
load_dotenv()

True

In [11]:
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage

model = init_chat_model("google_genai:gemini-2.5-flash")
model.invoke([HumanMessage("부산은 지금 몇시야?")])

AIMessage(content='지금 부산은 **2024년 5월 16일 목요일 오후 8시 39분**입니다. (한국 표준시 KST)', additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--62f232f4-8bae-401e-836d-469b7cbdbdfc-0', usage_metadata={'input_tokens': 8, 'output_tokens': 523, 'total_tokens': 531, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 487}})

#### ZoneInfo 간단 사용법

In [12]:
from datetime import datetime
from zoneinfo import ZoneInfo # uv add tzdata

# 1. ZoneInfo 객체 생성
seoul_tz = ZoneInfo("Asia/Seoul") # 해당 지역의 시간대 정보 (오프셋, 일광 절약 시간제 등)
newyork_tz = ZoneInfo("America/New_York")
print(seoul_tz)
print(newyork_tz)


Asia/Seoul
America/New_York


In [13]:
type(seoul_tz)

zoneinfo.ZoneInfo

In [14]:
# 2. 현재 시간 가져오기 (시간대 정보 적용)
now_seoul = datetime.now(tz=seoul_tz)
now_newyork = datetime.now(tz=newyork_tz)

print(f"서울 현재 시간: {now_seoul}")
print(f"뉴욕 현재 시간: {now_newyork}")

서울 현재 시간: 2026-04-05 20:40:32.033062+09:00
뉴욕 현재 시간: 2026-04-05 07:40:32.033062-04:00


#### 도구 생성

In [15]:
from langchain.tools import tool

@tool
def get_current_time(timezone: str, location: str) -> str:
    """ 현재 시각을 반환하는 함수

    Args:
        timezone (str): 타임존 (예: 'Asia/Seoul') 실제 존재하는 타임존이어야 함
        location (str): 지역명. 타임존이 모든 지명에 대응되지 않기 때문에 이후 llm 답변 생성에 사용됨
    """
    target_timezone = ZoneInfo(timezone)
    now = datetime.now(target_timezone).strftime("%Y-%m-%d %H:%M:%S")
    location_and_local_time = f'{timezone} ({location}) 현재시각 {now} ' # 타임존, 지역명, 현재시각을 문자열로 반환
    print(location_and_local_time)
    return location_and_local_time

In [16]:
from langchain.agents import create_agent

# 도구들을 tools 리스트에 추가
tools = [get_current_time,]

# 에이전트 생성
agent = create_agent(
    model,
    tools=tools,
    system_prompt="너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다."
)

In [17]:
result = agent.invoke(
    {"messages": [{"role": "user", "content": "부산은 지금 몇시야?"}]},
)

Asia/Seoul (부산) 현재시각 2026-04-05 20:40:33 


In [18]:
result

{'messages': [HumanMessage(content='부산은 지금 몇시야?', additional_kwargs={}, response_metadata={}, id='7062cdef-725e-4219-ad9c-876a3a035865'),
  AIMessage(content='', additional_kwargs={'function_call': {'name': 'get_current_time', 'arguments': '{"timezone": "Asia/Seoul", "location": "\\ubd80\\uc0b0"}'}, '__gemini_function_call_thought_signatures__': {'fd8e2ce2-12ef-4af2-a1b6-004e2a12411a': 'CpYEAb4+9vv757B9Duq3Y1nR7CqQ0Kx6rruV6zf1vH8aPSFzWsMdUo2leHhHf1O42+QARzAshBzdD7y9W/a4E61XXc64+5kRPTCEZ74cxgBF1TOEZbugFHsrxqEWUTQ9SSfO4ulTUK2VjxE+SLRKDOrwiOjQFlAPYz5P+W6Xor35ZWARToWfiOR7YtqiCtpprsTERI7KNjOe66hdMUbnK3tnyat5KcCxUu35Z9i6Fiyx42CCIXrQR2y5QV9LDIfPENd5BXIlJa7OxQHfQwsDifowM9bwIYa3axEO4+XK1sh8guI0Nv2hRQZwVUGz13ABAz+8+9iF0tcA28+s0OFhW9/mS78UAa/WqAXjtH2jUv5AD4K3Zohy8RnpdV/p0JkurZbLuK/ms/gPm62p9/woYG2SoP68G5pMyt2sa4flvOFUpwerS70ZJwKNADEMfpXTnuc+NnYCl9+Rb9167gPNp+MP59HhG/HNgMGIonnF+PZ15CKzHxEnSqr7XjxwJUj46N2NduD4a2vVe9iq6mUAxc7E7BhMaWjMX3Oj5RRokS4JmoREFPuZ4vobglh1fy8ZKcnlBf6ll+4rHeKXPwucNm9QaEwYL2s4zb